In [7]:
from pathlib import Path
import pandas as pd
import duckdb
import os
DATA_DIR = Path.cwd() / 'Data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'Data'

In [14]:
encounters_df = pd.read_parquet(DATA_DIR / 'encounters.parquet')
encounters_df = encounters_df.rename(columns={
    'PATIENT': 'patient_id',
    'START': 'encounter_date',
})
encounters_df['encounter_date'] = pd.to_datetime(encounters_df['encounter_date'])

In [12]:
stg_medications = pd.read_csv(
    Path.cwd() / 'staging' / 'medications.csv',
    parse_dates=['start_date', 'end_date'],
 )
int_symptom_observations = pd.read_csv(
    Path.cwd() / 'staging' / 'observations.csv',
 )

In [16]:
result = duckdb.sql("""
    SELECT
        patient_id,
        MIN(encounter_date) AS first_encounter_date,
        MAX(encounter_date) AS last_encounter_date,
        DATEDIFF(
            'day',
            MIN(encounter_date),
            MAX(encounter_date)
        ) AS days_active
    FROM encounters_df
    GROUP BY patient_id
    HAVING DATEDIFF(
        'day',
        MIN(encounter_date),
        MAX(encounter_date)
    ) > 365
    ORDER BY days_active DESC
""").df()

result

,patient_id,first_encounter_date,last_encounter_date,days_active
0,4024CD03-F601-4F9F-2A82-85076020FFFE,1947-03-21 00:39:24+05:30,2023-09-10 00:39:24+05:30,27932
1,6164A161-50FE-BD90-5751-F935BEEDCD98,1947-12-06 14:59:52+05:30,2023-07-10 14:59:52+05:30,27610
2,FE46F05E-46D8-450D-5778-56D5268498A6,1948-09-01 14:53:47+05:30,2023-11-17 14:53:47+05:30,27470
3,65CE0D6A-BDCB-53AF-A8BD-DED4B386FFAC,1949-04-11 14:37:51+05:30,2024-01-15 14:37:51+05:30,27307
4,3BA51040-7D6A-BDC8-56A6-A393899C4F86,1948-10-25 11:00:00+05:30,2023-04-20 11:00:00+05:30,27205
...,...,...,...,...
9840,C976896D-AD9E-C3D7-128A-578D142691D6,2022-09-06 00:15:49+05:30,2023-11-14 00:15:49+05:30,434
9841,21A1F737-349C-D9E7-5A2E-F820CDFE10A3,2022-11-02 12:52:08+05:30,2024-01-10 12:52:08+05:30,434
9842,47A5C99F-334A-5DAF-52CE-8312F0A46FB4,2022-08-22 01:47:07+05:30,2023-10-30 01:47:07+05:30,434
9843,3E4FE892-2D4B-D551-C3A0-C12E3F761A3A,2022-08-23 09:56:11+05:30,2023-10-31 09:56:11+05:30,434


In [13]:
medication_fatigue_result = duckdb.sql("""
    WITH active_medications AS (
        SELECT DISTINCT
            patient AS patient_id,
            medication_name,
            end_date
        FROM stg_medications
        WHERE end_date IS NULL
           OR end_date > CURRENT_TIMESTAMP
    ),

    patient_fatigue AS (
        SELECT
            patient_id,
            AVG(symptom_value) AS fatigue_score
        FROM int_symptom_observations
        WHERE LOWER(symptom_name) = 'fatigue'
        GROUP BY patient_id
    ),

    medication_fatigue AS (
        SELECT
            medications.medication_name,
            medications.patient_id,
            fatigue.fatigue_score
        FROM active_medications AS medications
        INNER JOIN patient_fatigue AS fatigue
            ON medications.patient_id = fatigue.patient_id
    )

    SELECT
        medication_name,
        ROUND(AVG(fatigue_score), 2) AS average_fatigue_score,
        COUNT(DISTINCT patient_id) AS patient_count
    FROM medication_fatigue
    GROUP BY medication_name
    HAVING COUNT(DISTINCT patient_id) >= 10
    ORDER BY medication_name
""").df()
medication_fatigue_result

,medication_name,average_fatigue_score,patient_count
0,NAPROXEN SODIUM 220 MG ORAL TABLET,31.15,66
1,Naproxen sodium 220 MG Oral Tablet,25.01,838
2,Vitamin B12 5 MG/ML Injectable Solution,25.10,50
3,cycloSPORINE modified 100 MG Oral Capsule,24.13,15
4,ferrous sulfate 325 MG Oral Tablet,22.48,50
5,naproxen sodium 220 mg oral tablet,25.91,44
6,predniSONE 20 MG Oral Tablet,25.29,17
